In [0]:
# ============================================
# Stage 4.5: Load DimCustomer (Most Complex)
# ============================================

from pyspark.sql import functions as F

base = "abfss://raw@stadventureworksdw.dfs.core.windows.net/AdventureWorks2019/"

df_customer     = spark.read.parquet(base + "Customer/")
df_person       = spark.read.parquet(base + "Person/")
df_store        = spark.read.parquet(base + "Store/")
df_bea          = spark.read.parquet(base + "BusinessEntityAddress/")
df_address      = spark.read.parquet(base + "Address/")
df_stateprov    = spark.read.parquet(base + "StateProvince/")
df_countryreg   = spark.read.parquet(base + "CountryRegion/")
df_addresstype  = spark.read.parquet(base + "AddressType/")

print(f"Customer: {df_customer.count()}")
print(f"Person: {df_person.count()}")
print(f"Store: {df_store.count()}")
print(f"BusinessEntityAddress: {df_bea.count()}")

Customer: 19820
Person: 19972
Store: 701
BusinessEntityAddress: 19614


In [0]:
# Part 1: Individual 客户（含地址）

from pyspark.sql import functions as F

# 先筛选 Home 地址
home_addresses = (df_bea
    .join(df_addresstype, on="AddressTypeID", how="inner")
    .filter(F.col("Name") == "Home")
    .select("BusinessEntityID", "AddressID")
    .dropDuplicates(["BusinessEntityID"])
)

# 个人客户与 Person 表 JOIN（用别名避免字段冲突）
individual = (df_customer.alias("c")
    .join(df_person.alias("p"), F.col("c.PersonID") == F.col("p.BusinessEntityID"), how="inner")
    .join(home_addresses.alias("ha"), F.col("c.PersonID") == F.col("ha.BusinessEntityID"), how="left")
    .join(df_address.alias("a"), F.col("ha.AddressID") == F.col("a.AddressID"), how="left")
    .join(df_stateprov.alias("sp"), F.col("a.StateProvinceID") == F.col("sp.StateProvinceID"), how="left")
    .join(df_countryreg.alias("cr"), F.col("sp.CountryRegionCode") == F.col("cr.CountryRegionCode"), how="left")
    .select(
        F.col("c.CustomerID").cast("int").alias("CustomerID"),
        F.lit("Individual").alias("CustomerType"),
        F.concat_ws(" ", F.col("p.FirstName"), F.col("p.LastName")).alias("CustomerName"),
        F.col("a.City").alias("City"),
        F.col("sp.Name").alias("StateProvince"),
        F.col("cr.Name").alias("Country")
    )
)

print(f"Individual rows: {individual.count()}")

Individual rows: 19119


In [0]:
# Part 2: Store 客户

main_office = (df_bea
    .join(df_addresstype, on="AddressTypeID", how="inner")
    .filter(F.col("Name") == "Main Office")
    .select("BusinessEntityID", "AddressID")
    .dropDuplicates(["BusinessEntityID"])
)

store = (df_customer.alias("c")
    .filter((F.col("c.PersonID").isNull()) & (F.col("c.StoreID").isNotNull()))
    .join(df_store.alias("s"), F.col("c.StoreID") == F.col("s.BusinessEntityID"), how="inner")
    .join(main_office.alias("mo"), F.col("c.StoreID") == F.col("mo.BusinessEntityID"), how="left")
    .join(df_address.alias("a"), F.col("mo.AddressID") == F.col("a.AddressID"), how="left")
    .join(df_stateprov.alias("sp"), F.col("a.StateProvinceID") == F.col("sp.StateProvinceID"), how="left")
    .join(df_countryreg.alias("cr"), F.col("sp.CountryRegionCode") == F.col("cr.CountryRegionCode"), how="left")
    .select(
        F.col("c.CustomerID").cast("int").alias("CustomerID"),
        F.lit("Store").alias("CustomerType"),
        F.col("s.Name").alias("CustomerName"),
        F.col("a.City").alias("City"),
        F.col("sp.Name").alias("StateProvince"),
        F.col("cr.Name").alias("Country")
    )
)

print(f"Store rows: {store.count()}")

Store rows: 701


In [0]:
# UNION + 添加 SCD 字段和代理键
dim_customer = (individual
    .unionByName(store)
    .withColumn("CustomerKey", F.monotonically_increasing_id() + 1)
    .withColumn("valid_from", F.current_timestamp())
    .withColumn("valid_to", F.to_date(F.lit("9999-12-31")))
    .withColumn("is_current", F.lit(1).cast("int"))
    .select("CustomerKey", "CustomerID", "CustomerType", "CustomerName",
            "City", "StateProvince", "Country",
            "valid_from", "valid_to", "is_current")
)

print(f"Total DimCustomer rows: {dim_customer.count()}")

Total DimCustomer rows: 19820


In [0]:
# 写入 Curated Zone
curated_path = "abfss://curated@stadventureworksdw.dfs.core.windows.net/AdventureWorksDW/DimCustomer/"

dim_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .save(curated_path)

print(f"✅ DimCustomer written to {curated_path}")

✅ DimCustomer written to abfss://curated@stadventureworksdw.dfs.core.windows.net/AdventureWorksDW/DimCustomer/


In [0]:
# 验证
df_check = spark.read.format("delta").load(curated_path)
print(f"Total rows: {df_check.count()}")
# 期望：19820

df_check.groupBy("CustomerType").count().show()
# 期望：Individual = 19119, Store = 701

Total rows: 19820
+------------+-----+
|CustomerType|count|
+------------+-----+
|  Individual|19119|
|       Store|  701|
+------------+-----+

